# 14 Case Study — Exercise

Using the Pine and Cypress Nursing Home Legionnaires' disease data, independently produce a mini outbreak investigation report.

In [ ]:
# Google Colab setup -- skip this cell if running locally
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || True
    os.chdir('/content/python4epi')
    !pip install -q -e .

In [ ]:
import pathlib

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from scipy import stats

# -- CJK font setup (prevents CJK labels from rendering as boxes) --
# Scan system font directories and explicitly register CJK fonts (more reliable than relying on cache)
for _font_dir in map(pathlib.Path, ["/usr/share/fonts", "/usr/local/share/fonts"]):
    if _font_dir.exists():
        for _fp in sorted(_font_dir.rglob("*")):
            if _fp.suffix.lower() in {".ttf", ".ttc", ".otf"} and (
                "CJK" in _fp.name or "WenQuanYi" in _fp.name or "wqy" in _fp.name
            ):
                try:
                    fm.fontManager.addfont(str(_fp))
                except Exception:
                    pass

plt.rcParams["font.sans-serif"] = [
    "Noto Sans CJK TC", "Noto Sans CJK SC", "Noto Sans CJK JP",
    "Noto Sans TC", "Microsoft JhengHei",
    "WenQuanYi Zen Hei", "SimHei", "Arial Unicode MS",
    "Heiti TC", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False
plt.style.use("ggplot")
plt.rcParams["figure.dpi"] = 150


## Question 1: Outbreak summary table

1. Read `data/synthetic/legionella_outbreak.csv`
2. Calculate the following metrics:
   - Total residents, number infected, number of deaths
   - Attack rate, case fatality rate
   - Hospitalization rate (hospitalized / infected)
   - ICU proportion (ICU / hospitalized)
3. Assemble the results into a table using `pd.DataFrame`

In [ ]:
# TODO: read the data
# TODO: calculate each metric
# TODO: assemble into a summary table

## Question 2: Quick risk-factor screening

For each of the following 4 exposure factors, build a 2x2 table and calculate the RR:
- `shower_use`
- `hydrotherapy_use`
- `comorbidity_copd`
- `immunosuppressed`

Compute them all in one `for` loop and output a comparison table.
Which factor has the largest RR?

In [ ]:
# TODO: compute the RR for the 4 factors using a for loop
# TODO: output the comparison table
# TODO: find the factor with the largest RR

## Question 3 (Challenge): Mini SitRep

Produce a mini SitRep containing the following 3 charts:

1. **Epidemic curve**: daily new case counts (bar chart), marking the peak day
2. **Age distribution**: age histogram of infected vs. not infected
3. **Attack rate by floor**: bar chart of attack rates for the 6 zones (1F-A ~ 3F-B)

Lay them out in a single row with `fig, axes = plt.subplots(1, 3)`.
At the end, print the action recommendations -- based on your analysis, which zone should be addressed first?

In [ ]:
# TODO: draw the 3 charts
# TODO: print the action recommendations

## Question 4: Norovirus banquet cluster mini outbreak investigation (Norovirus scenario)

A gastroenteritis cluster breaks out after a banquet. Conduct a complete mini outbreak investigation from the guests' food exposure and onset line list.

1. Plot the epidemic curve (onset time) and determine the transmission pattern
2. Calculate the attack rate and risk ratio (RR) for each food item
3. Identify the suspect food with the highest RR and run a chi-square test
4. Write a conclusion: the suspected source of infection and what the epidemic curve shape implies

In [ ]:
# Norovirus banquet cluster: food exposure and onset line list for 150 guests
from epi_learning.metrics import attack_rate, risk_ratio
rng = np.random.default_rng(1404)
n = 150
foods = ["生蠔", "沙拉", "甜點", "湯品"]
ate = {f: rng.binomial(1, 0.5, n) for f in foods}
p_ill = (0.05 + 0.7 * ate["生蠔"]).clip(0, 1)     # oysters are contaminated
ill = rng.binomial(1, p_ill)
onset_hr = np.where(ill == 1, rng.normal(32, 8, n).clip(6, 72), np.nan)  # norovirus incubation ~24-48h
guests = pd.DataFrame({"guest_id": range(1, n + 1), "ill": ill,
                       **{f: ate[f] for f in foods}, "onset_hr": np.round(onset_hr, 0)})
print(f"Banquet: {n} guests, {ill.sum()} ill ({ill.mean():.1%})")

# TODO: plot the epidemic curve -- bin ill guests' onset_hr into 6-hour intervals and determine whether it's a point source
# TODO: for each food, calculate the attack rate and risk ratio (RR) for "ate vs did not eat" (use risk_ratio)
# TODO: find the suspect food with the highest RR, and test its significance with a scipy.stats chi-square test
# TODO: write a conclusion: what is the suspected source of infection? what transmission pattern does the epidemic curve shape support?

## Question 5: COVID-19 workplace cluster investigation (COVID-19 scenario)

A company has a COVID-19 cluster, and a company-wide meeting is suspected to be the exposure event.

1. Plot the epidemic curve (by onset day)
2. Calculate the attack rate by department and identify the highest-risk department
3. Calculate the risk ratio (RR) for "attended the meeting"
4. Write a conclusion: was the meeting a suspected exposure?

In [ ]:
# COVID-19 workplace cluster: 200 employees at a company, a company-wide meeting as the suspected exposure
from epi_learning.metrics import risk_ratio
rng = np.random.default_rng(1405)
n = 200
dept = rng.choice(["業務", "研發", "行政", "客服"], n, p=[0.3, 0.3, 0.2, 0.2])
meeting = rng.binomial(1, np.where(dept == "業務", 0.9, 0.4))   # sales staff mostly attended
p_inf = (0.03 + 0.35 * meeting).clip(0, 1)
infected = rng.binomial(1, p_inf)
onset_day = np.where(infected == 1, rng.integers(2, 10, n), -1)   # days after the meeting when symptoms started
staff = pd.DataFrame({"emp_id": range(1, n + 1), "dept": dept,
                      "meeting": meeting, "infected": infected, "onset_day": onset_day})
print(f"Company: {n} employees, {infected.sum()} confirmed cases; {meeting.sum()} attended the meeting")

# TODO: plot the epidemic curve (tally daily new cases by onset_day)
# TODO: use groupby to calculate the confirmed case count and attack rate by department, and identify the highest-risk department
# TODO: calculate the risk ratio (RR) for "attended the meeting vs did not attend"
# TODO: write a conclusion: was the meeting a suspected exposure event? why does the sales department have the highest risk?

## Question 6 (Challenge): Dengue fever community outbreak SitRep (Dengue fever scenario)

A community is experiencing a dengue fever epidemic, and you need to produce a situation report (SitRep).

1. Plot the community-wide weekly epidemic curve
2. Calculate the cumulative incidence rate per 100,000 population for each district and rank the hotspots
3. Determine the epidemic trend (rising / stable / declining)
4. Write a 3-5 sentence SitRep: scale, hotspots, trend, and prevention recommendations

In [ ]:
# Dengue fever community outbreak: weekly case counts and population for 5 districts over 10 weeks (Challenge: write a SitRep)
rng = np.random.default_rng(1406)
districts = ["安南區", "三民區", "北屯區", "板橋區", "中西區"]
pop = {"安南區": 190000, "三民區": 340000, "北屯區": 280000, "板橋區": 550000, "中西區": 78000}
weekly_rate = {"安南區": 3.0, "三民區": 1.2, "北屯區": 0.8, "板橋區": 0.6, "中西區": 1.4}  # per 100k/week
_rows = []
for wk in range(1, 11):
    growth = 1.0 + 0.15 * wk    # epidemic growing week over week
    for d in districts:
        cases = rng.poisson(weekly_rate[d] * growth * pop[d] / 100000)
        _rows.append({"epi_week": wk, "district": d, "cases": cases})
dengue = pd.DataFrame(_rows)
region_pop = pd.DataFrame({"district": districts, "population": [pop[d] for d in districts]})
print(f"Dengue fever: {dengue['cases'].sum()} cases, 10 weeks x {len(districts)} districts")

# TODO: plot the community-wide weekly epidemic curve (groupby epi_week and sum)
# TODO: merge with region_pop, calculate the "cumulative incidence rate per 100,000" for each district, and rank the hotspots
# TODO: use the weekly case counts to determine the epidemic trend (rising / stable / declining)
# TODO: write a 3-5 sentence SitRep: current scale, hotspots, trend, and recommended prevention actions